# Toy stacked-GP hierarchy (Damianou & Lawrence 2013, §4.1 / fig. 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/01_toy_hierarchy.ipynb)

**Paper:** [Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215).](https://proceedings.mlr.press/v31/damianou13a.html)

Generate observations from a three-level stack of GPs (top latent → 2D intermediate → 10D data), fit an unsupervised DeepGP with overcomplete ARD widths, and compare recovered signals to stacked PCA. In the paper, only the deep GP recovers latent signals close to the ground truth while learning dimensionalities via ARD.


In [ ]:
# Install dependencies when running on Google Colab
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    pkgs = ["numpy>=1.7", "scipy>=0.12", "matplotlib", "GPy>=1.15"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/SheffieldML/PyDeepGP.git@main",
    ])
    print("Installed GPy and PyDeepGP for Colab.")
else:
    print("Local environment — ensure PyDeepGP and GPy are importable.")


In [ ]:
# Set QUICK=True for a short smoke run (fewer iterations / smaller models).
# Set QUICK=False for a paper-scale demonstration (slower).
QUICK = True
SEED = 42


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import GPy
import deepgp

np.random.seed(SEED)

def sample_gp_prior(X, kernel, output_dim, noise=1e-6):
    K = kernel.K(X) + noise * np.eye(X.shape[0])
    L = np.linalg.cholesky(K)
    return L @ np.random.randn(X.shape[0], output_dim)

def rbf_bias(q, ard=True):
    return GPy.kern.RBF(q, ARD=ard) + GPy.kern.Bias(q)

def ard_weights(kern):
    if hasattr(kern, "rbf"):
        ls = np.asarray(kern.rbf.lengthscale).ravel()
    else:
        ls = np.asarray(kern.lengthscale).ravel()
    return 1.0 / np.maximum(ls, 1e-12)

def dominant_dims(w, k=2):
    return list(np.argsort(-w)[:k])


## Generative hierarchy

Top latent $Z \in \mathbb{R}$ drives a 2D intermediate layer, which drives 10D observations (paper fig. 2a).


In [ ]:
n = 40 if QUICK else 80
Z = np.linspace(-1.0, 1.0, n)[:, None]
X_true = sample_gp_prior(Z, GPy.kern.RBF(1, lengthscale=0.4), output_dim=2)
X_true += 0.02 * np.random.randn(*X_true.shape)
Y = sample_gp_prior(X_true, GPy.kern.RBF(2, lengthscale=0.8), output_dim=10)
Y += 0.05 * np.random.randn(*Y.shape)

fig, ax = plt.subplots(figsize=(7, 3))
t = np.arange(n)
ax.plot(t, Z[:, 0], "k-", label="top $Z$")
ax.plot(t, X_true[:, 0], label="mid 0")
ax.plot(t, X_true[:, 1], label="mid 1")
ax.set_title("Ground-truth hierarchy (fig. 2a style)")
ax.legend()
plt.show()
print("Y", Y.shape)


## Fit DeepGP

Overcomplete latent widths; ARD should down-weight unused dimensions (paper: dimensionalities learnt automatically).


In [ ]:
q_mid, q_top = (4, 3) if QUICK else (5, 4)
num_inducing = 15 if QUICK else 25
iters_fixed, iters_free = (50, 150) if QUICK else (200, 1500)

from GPy.core.parameterization.variational import NormalPosterior, VariationalPosterior

def out_var(y):
    if isinstance(y, (NormalPosterior, VariationalPosterior)):
        return float(np.asarray(y.mean).var())
    return float(np.asarray(y).var())

m = deepgp.DeepGP(
    [Y.shape[1], q_mid, q_top],
    Y,
    kernels=[rbf_bias(q_mid), rbf_bias(q_top)],
    num_inducing=num_inducing,
    back_constraint=False,
)
for layer in m.layers:
    layer.Gaussian_noise.variance = out_var(layer.Y) * 0.01
    layer.Gaussian_noise.variance.fix()
m.optimize(max_iters=iters_fixed, messages=False)
for layer in m.layers:
    layer.Gaussian_noise.variance.unfix()
m.optimize(max_iters=iters_free, messages=True)
print(f"Variational bound: {float(m.log_likelihood()):.3f}")
for i, layer in enumerate(m.layers):
    print(f"Layer {i} ARD:", np.array2string(ard_weights(layer.kern), precision=3))


## Recovered latents vs stacked PCA


In [ ]:
def pca_project(Y, q):
    Yc = Y - Y.mean(0)
    U, S, _ = np.linalg.svd(Yc, full_matrices=False)
    q = min(q, U.shape[1])
    return U[:, :q] * S[:q]

pca_mid = pca_project(Y, q_mid)
pca_top = pca_project(pca_mid, q_top)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].plot(t, Z[:, 0], "k-", label="top")
axes[0].plot(t, X_true[:, 0], label="mid0")
axes[0].plot(t, X_true[:, 1], label="mid1")
axes[0].set_title("Ground truth")
axes[0].legend(fontsize=8)

for ax, li, title in ((axes[1], 0, "DeepGP near data"), (axes[2], 1, "DeepGP top")):
    Xm = m.layers[li].X.mean
    for d in dominant_dims(ard_weights(m.layers[li].kern), k=min(2, Xm.shape[1])):
        ax.plot(t, Xm[:, d], label=f"dim {d}")
    ax.set_title(title)
    ax.legend(fontsize=8)
fig.suptitle("Fig. 2 style — DeepGP recovery")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, pca_mid[:, 0], label="PCA mid 0")
if pca_mid.shape[1] > 1:
    ax.plot(t, pca_mid[:, 1], label="PCA mid 1")
ax.plot(t, pca_top[:, 0], label="PCA top 0")
ax.set_title("Stacked PCA (greedy)")
ax.legend(fontsize=8)
plt.show()


## References

- Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215). — [PDF](https://proceedings.mlr.press/v31/damianou13a.html)
- Script counterpart: `examples/damianou2013/toy_hierarchy.py`
